In [2]:
n_t1 = len(list((DATA_DIR/"t1").glob("*T1w_MNI*.nii.gz")))
n_mk = len(list((DATA_DIR/"masks").glob("*lesion_mask_MNI*.nii.gz")))
print("train_hires counts → T1:", n_t1, " | masks:", n_mk)


train_hires counts → T1: 139  | masks: 139


In [1]:
from pathlib import Path
import importlib.util, os
from tensorflow.keras import mixed_precision



# 2) mixed precision
mixed_precision.set_global_policy("mixed_float16")

# 3) paths
MODULE_PATH = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seg)

RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined")
DATA_DIR = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires")
MODEL_DIR = RUN_ROOT / "models"
CALLBACKS_DIR = RUN_ROOT / "callbacks"
LOG_DIR = RUN_ROOT / "logs"
for d in (MODEL_DIR, CALLBACKS_DIR, LOG_DIR): d.mkdir(parents=True, exist_ok=True)
os.environ["SMARTSOTA_LOG_DIR"] = str(LOG_DIR)

# 4) Kick off training with memory-friendly overrides
history = seg.train_dynamic_model(
    DATA_DIR=DATA_DIR,          # single-folder with all .nii.gz
    MODEL_DIR=MODEL_DIR,
    CALLBACKS_DIR=CALLBACKS_DIR,
    TOTAL_EPOCHS=60,
    VALIDATION_SPLIT=0.10,
    BATCH_SIZE=2,               # ↓ memory
    BASE_FILTERS=8,             # ↓ memory
    RESAMPLE_TO_TARGET=True,
    INPUT_SHAPE=(192, 224, 192, 1),   # optional: slightly smaller than 208×240×208
)


2025-11-04 15:39:21.898494: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1762295963.809674  364305 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1762295963.810818  364305 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22148 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:41:00.0, compute capability: 8.9
I0000 00:00:1762295963.811152  364305 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 1
I0000 00:00:1762295963.812228  364305 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22122 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:61:00.0, compute capability: 8.9
2025-11-04 15:39:23,873 - SmartSOTA_Dynamic - INFO - ✅ All imports successful
2025-11-04 15:39:23,873 - SmartSOTA_Dynamic - INFO - TensorFlow eager execution: True
2025-11-04 15:39:23,873 - SmartSOTA_Dynamic - INFO - Environment verified:
- Python 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
- TensorFlo

Strategy: MirroredStrategy


2025-11-04 15:39:24,389 - SmartSOTA_Dynamic - INFO - 📐 Detected max volume dimensions: (193, 229, 193) → rounded up to: (208, 240, 208)
2025-11-04 15:39:24,390 - SmartSOTA_Dynamic - INFO - 🧭 INPUT_SHAPE set to: (208, 240, 208, 1)
2025-11-04 15:39:24,390 - SmartSOTA_Dynamic - INFO - 📚 Loading dataset (flex loader for T1w volumes)…
2025-11-04 15:39:24,391 - SmartSOTA_Dynamic - INFO - Memory at dataset_load_start: CPU=0.91GB | GPU mem tracking failed | Disk: 1255.1GB free
2025-11-04 15:39:24,395 - SmartSOTA_Dynamic - INFO - 📁 Single-folder mode: 323 images, 323 masks in /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized/_combined_manifests_2bins_normal/_combined_splits_70_30_global/train_hires
2025-11-04 15:39:24,395 - SmartSOTA_Dynamic - INFO - Found 323 image files and 323 mask files
2025-11-04 15:40:06,112 - SmartSOTA_Dynamic - INFO - 📊 Created 323 image–mask pairs
2025-11-04 15:40:06,113 - SmartSOTA_Dynamic - INFO - 🧠 Lesion presence: 10

...

 78/129 ━━━━━━━━━━━━━━━━━━━━ 34s 676ms/step - dice_coefficient: 0.1154 - loss: 1.0600

2025-11-04 15:56:05,886 - SmartSOTA_Dynamic - INFO - Memory at batch_1240: CPU=8.00GB | GPU mem tracking failed | Disk: 1255.1GB free


 79/129 ━━━━━━━━━━━━━━━━━━━━ 34s 680ms/step - dice_coefficient: 0.1156 - loss: 1.0598

2025-11-04 15:56:06.046929: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:361] gpu_async_0 cuMemAllocAsync failed to allocate 332267520 bytes: RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
 Reported by CUDA: Free memory/Total memory: 323026944/25261047808
2025-11-04 15:56:06.046957: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:366] Stats: Limit:                     23223926784
InUse:                     14322964980
MaxInUse:                  19465772666
NumAllocs:                     4934486
MaxAllocSize:               5665975920
Reserved:                            0
PeakReserved:                        0
LargestFreeBlock:                    0

2025-11-04 15:56:06.047024: E external/local_xla/xla/stream_executor/gpu/gpu_cudamallocasync_allocator.cc:70] Histogram of current allocation: (allocation_size_in_bytes, nb_allocation_of_that_sizes), ...;
2025-11-04 15:56:06.047027: E external/local_xla/xla/stream_

ResourceExhaustedError: Graph execution error:

Detected at node gradient_tape/SmartSOTA_Dynamic_1/vision_mamba_block_7_1/layer_normalization_34_1/mul_1/Mul_1 defined at (most recent call last):
  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/threading.py", line 973, in _bootstrap

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/threading.py", line 1016, in _bootstrap_inner

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/site-packages/keras/src/backend/tensorflow/trainer.py", line 78, in train_step

Detected at node gradient_tape/SmartSOTA_Dynamic_1/vision_mamba_block_7_1/layer_normalization_34_1/mul_1/Mul_1 defined at (most recent call last):
  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/threading.py", line 973, in _bootstrap

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/threading.py", line 1016, in _bootstrap_inner

  File "/home/rbielski/miniconda3/envs/tf_310/lib/python3.10/site-packages/keras/src/backend/tensor